# LLM Classification Finetuning

## 1. Train

In [1]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import torch
import evaluate

LLM_MODEL = "distilbert-base-uncased"
train_data_path = "kaggle/input/competitions/llm-classification-finetuning/train.csv"
test_data_path = "kaggle/input/competitions/llm-classification-finetuning/test.csv"
model_save_path = "./model"
data_submission_path = "submission.csv"


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Data loading + preprocessing
df = pd.read_csv(train_data_path)

# 1) Label encoding
if set(["winner_model_a", "winner_model_b", "winner_tie"]).issubset(df.columns):
    df["labels"] = (
        df["winner_model_a"] * 0 +
        df["winner_model_b"] * 1 +
        df["winner_tie"] * 2
    ).astype("int64")
else:
    raise ValueError("Missing one of winner_model_a/winner_model_b/winner_tie")

# 2) Build text field
if set(["prompt", "response_a", "response_b"]).issubset(df.columns):
    df["text"] = (
        "Prompt:\n" + df["prompt"].astype(str) +
        "\n\nResponse A:\n" + df["response_a"].astype(str) +
        "\n\nResponse B:\n" + df["response_b"].astype(str)
    )
else:
    raise ValueError("Missing one of prompt/response_a/response_b")

print("Loaded", len(df), "rows")
print(df["labels"].value_counts())

Loaded 57477 rows
labels
0    20064
1    19652
2    17761
Name: count, dtype: int64


In [3]:


# 3) HF dataset
dataset = Dataset.from_pandas(df[["text", "labels"]])

# 4) split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]


def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )


tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


Map: 100%|██████████| 5748/5748 [00:00<00:00, 7188.76 examples/s]


In [4]:


model = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL,
    num_labels=3
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 21003.02it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


training_args = TrainingArguments(
    output_dir=model_save_path,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=1200,
    warmup_steps=0.1,
    weight_decay=0.01,
    fp16=False,
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=120,
    save_steps=480,
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    logging_nan_inf_filter=False,
    # use_cpu=True,
)


In [6]:


data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics, 
)

In [7]:
#B. Check a manual forward pass on a train batch

# Do this before full training and then again after a few steps:

name, p = next((n, p) for n, p in model.named_parameters() if p.requires_grad)
before = p.detach().cpu().clone()

trainer.train()

after = dict(model.named_parameters())[name].detach().cpu()
print("param changed?", not torch.equal(before, after))
print("max abs diff:", (after - before).abs().max().item())

/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
120,4.381438,1.095979,0.350905
240,4.383588,1.090193,0.374391
360,4.399827,1.090603,0.364301
480,4.338418,1.089330,0.367954
600,4.342097,1.089344,0.364997
720,4.352793,1.092982,0.382394
840,4.291396,1.085498,0.402401
960,4.438096,1.086416,0.394224
1080,4.333079,1.081671,0.409708
1200,4.373098,1.080350,0.407620


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.28it/s]
/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, dev

param changed? True
max abs diff: 0.002167031168937683


In [8]:
trainer.remove_callback(type(trainer.callback_handler.callbacks[-1]))  # not ideal/generic

In [9]:
print("global_step:", trainer.state.global_step)
print(trainer.state.log_history[-10:])
print("val metrics:", trainer.evaluate(val_dataset))

pred = trainer.predict(val_dataset)
print("nan/inf", np.any(np.isnan(pred.predictions)), np.any(np.isinf(pred.predictions)))

global_step: 1200
[{'loss': 4.374279479980469, 'grad_norm': 1.8321104049682617, 'learning_rate': 2.7962962962962963e-06, 'epoch': 0.16238159675236807, 'step': 1050}, {'loss': 4.333079223632812, 'grad_norm': 1.8194122314453125, 'learning_rate': 2.3333333333333336e-06, 'epoch': 0.16624782524647205, 'step': 1075}, {'eval_loss': 1.081671118736267, 'eval_accuracy': 0.40970772442588727, 'eval_runtime': 27.4734, 'eval_samples_per_second': 209.221, 'eval_steps_per_second': 26.171, 'epoch': 0.16702107094529287, 'step': 1080}, {'loss': 4.350977783203125, 'grad_norm': 2.2828240394592285, 'learning_rate': 1.8703703703703705e-06, 'epoch': 0.17011405374057606, 'step': 1100}, {'loss': 4.306510009765625, 'grad_norm': 1.9744226932525635, 'learning_rate': 1.4074074074074075e-06, 'epoch': 0.17398028223468007, 'step': 1125}, {'loss': 4.301456298828125, 'grad_norm': 2.1186063289642334, 'learning_rate': 9.444444444444445e-07, 'epoch': 0.17784651072878407, 'step': 1150}, {'loss': 4.353592529296875, 'grad_nor

/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


val metrics: {'eval_loss': 1.0803498029708862, 'eval_accuracy': 0.40762004175365346, 'eval_runtime': 27.3333, 'eval_samples_per_second': 210.293, 'eval_steps_per_second': 26.305, 'epoch': 0.18557896771699206}


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


nan/inf False False


In [10]:
# Save the trained model and tokenizer
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 15.90it/s]


('./model/tokenizer_config.json', './model/tokenizer.json')

In [11]:

# Evaluate and show metrics
metrics = trainer.evaluate(eval_dataset=val_dataset)
print("Validation metrics:", metrics)

# Predict to verify output shape
predictions = trainer.predict(val_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)
print("Sample predictions:", pred_labels[:10])

/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Validation metrics: {'eval_loss': 1.0803498029708862, 'eval_accuracy': 0.40762004175365346, 'eval_runtime': 27.7476, 'eval_samples_per_second': 207.153, 'eval_steps_per_second': 25.912, 'epoch': 0.18557896771699206}


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Sample predictions: [0 0 1 0 0 2 2 1 1 1]


## 2.Predict

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(model_save_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_save_path)

model.eval()

Using device: cpu


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 26238.05it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [13]:
# 1) Load test data
test_df = pd.read_csv(test_data_path)

# 2) Prepare text in same format as training
test_df["text"] = (
    "Prompt:\n" + test_df["prompt"].astype(str) +
    "\n\nResponse A:\n" + test_df["response_a"].astype(str) +
    "\n\nResponse B:\n" + test_df["response_b"].astype(str)
)

test_dataset = Dataset.from_pandas(test_df[["text"]])

test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["input_ids", "attention_mask", "text"]])



Map: 100%|██████████| 3/3 [00:00<00:00, 950.59 examples/s]


In [15]:


# 4) Predict
preds = trainer.predict(test_dataset)
print("Raw predictions shape:", preds)
logits = preds.predictions  # shape (N, 3)
print("Logits shape:", logits)
logits_tensor = torch.from_numpy(logits).float()

if torch.isnan(logits_tensor).any() or torch.isinf(logits_tensor).any():
    raise ValueError("Logits contain NaN or Inf; check model/prediction pipeline")

# stable softmax to avoid numeric instabilities
logits_stable = logits_tensor - logits_tensor.max(dim=-1, keepdim=True).values
exp_scores = torch.exp(logits_stable)
probs_tensor = exp_scores / exp_scores.sum(dim=-1, keepdim=True)
probs = probs_tensor.cpu().numpy()

# 5) save to submission file
out = pd.DataFrame({
    "id": test_df["id"].values,
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_model_tie": probs[:, 2],
})

out.to_csv(data_submission_path, index=False)
print(out.head())


Raw predictions shape: PredictionOutput(predictions=array([[-0.30187875, -0.18388398,  0.4497218 ],
       [-0.06172824,  0.10229132, -0.22769107],
       [ 0.06742098, -0.06075042, -0.16913022]], dtype=float32), label_ids=None, metrics={'test_runtime': 0.0644, 'test_samples_per_second': 46.614, 'test_steps_per_second': 15.538})
Logits shape: [[-0.30187875 -0.18388398  0.4497218 ]
 [-0.06172824  0.10229132 -0.22769107]
 [ 0.06742098 -0.06075042 -0.16913022]]
        id  winner_model_a  winner_model_b  winner_model_tie
0   136060        0.235536        0.265034          0.499429
1   211333        0.330544        0.389459          0.279997
2  1233961        0.374665        0.329594          0.295740


/Users/yanxinye/github/ml-ops-notes/llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
